In [ ]:
### import dependencies
import mne, os, pickle, glob
import numpy as np
import pandas as pd
from os.path import join
import matplotlib
%gui qt
mne.set_log_level(verbose='WARNING')
matplotlib.use('Qt5Agg')

In [ ]:
## convert KIT file (.sqd) into Neuromag file (.fif)
expt = 'EXPT' # experiment name as written on the -raw.fif
ROOT = f'/path/to/{expt}' # change path to where your data is stored
RAW = join(ROOT, 'raw')
os.chdir(RAW) #setting current dir
subj = 'subj'
sess = 'SESSION'
date = '4.13.25'
marker_date = '250413'

# define file names
sqd_fname  = join(RAW, '%s_%s_%s/%s_%s_%s_NR.sqd')%(subj, expt, date, subj, sess, date) # raw KIT file with noise reduction
elp_fname  = join(RAW, '%s_%s_%s/%s.%s_Points.txt')%(subj, expt, date, subj, date)      # fiducial file from fastscan
hsp_fname  = join(RAW, '%s_%s_%s/%s.%s_HS.txt')%(subj, expt, date, subj, date)          # headshape file from fastscan
mrk1_fname = join(RAW, '%s_%s_%s/%s-1.mrk')%(subj, expt, date, marker_date)           # pre-exp marker
mrk2_fname = join(RAW, '%s_%s_%s/%s-2.mrk')%(subj, expt, date, marker_date)           # post-exp marker
mrks = [mrk1_fname, mrk2_fname]

# read kit raw file
raw = mne.io.read_raw_kit(sqd_fname,
                          mrk = mrks,
                          elp = elp_fname,
                          hsp = hsp_fname,
                          slope = '+',
                          stim_code= 'channel',
                          verbose = True,
                          preload = True)

# save it as fif file
fname = join(ROOT, 'data/meg/%s/%s_%s-raw.fif')%(subj, subj, sess) # filenames should end with -raw.fif (common raw data)
raw.save(fname, overwrite=True)
events = mne.find_events(raw, min_duration=0.002)
print(np.unique(events[:,2]))  # Print all unique event codes in the data
print(events) 

In [ ]:
#========= Parameters =========#
# STC parameters
SNR = 3 # Usually the rule of thumb has been to set to 3 for ANOVAs, 2 for single trial analyses. 
method = "dSPM"
fixed = False # set to True if you want signed data. this will make this command run: mne.convert_forward_solution(fwd, surf_ori=True)
n_jobs = 4
lambda2 = 1.0 / SNR ** 2.0

#===========================#
expt = 'EXPT' # experiment name as written on the -raw.fif
ROOT = f'/path/to/{expt}' # change path to where your data is stored
os.chdir(ROOT)
# recommended folder structure.
epochs_dir = join(ROOT, 'epochs/')
evokeds_dir = join(ROOT, 'evokeds/')
subjects_dir = join(ROOT,'mri/') # MRI directory, only needed for source level stuff
raw_dir = join(ROOT, 'raw/')
meg_dir = join(ROOT, 'meg/')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc/')

excluded = ['R0000']
## Use this when processing all the subjects
subjects = [i[:5] for i in os.listdir(raw_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")] # List of subjects, can also be added manually
## Use this when processing some of the subjects
print('N =', len(subjects))
print(subjects)

In [ ]:
### Data Loading Functions
def get_raw_data(subj):
    subj_dir = os.path.join(meg_dir, subj)
    raw_fname = os.path.join(subj_dir, '%s_%s-raw.fif') % (subj, sess)
    raw = mne.io.read_raw_fif(raw_fname, preload=True)

    return raw

def get_log_file(subj):
    log_filename = os.path.join(log_dir, subj, '%s_logfile.csv' % subj)
    log = pd.read_csv(log_filename)
            
    return log

def get_ica_data(subj):
    subj_dir = os.path.join(meg_dir, subj)
    raw_fname = os.path.join(subj_dir, '%s_%s-ica-raw.fif') % (subj, sess)
    raw = mne.io.read_raw_fif(raw_fname, preload=True)

    return raw

In [ ]:
### Combining Raw Files (if more than one)
for subj in subjects: # use 'for subj in subjects:' if you want to process all subjects at once
    subj_dir = os.path.join(meg_dir, subj)
    
    raw_fname = os.path.join(subj_dir, '%s_%s-raw.fif') % (subj, sess)
    raw1 = mne.io.read_raw_fif(raw_fname, preload=True)
    
    raw_fname = os.path.join(subj_dir, '%s_%s_2-raw.fif') % (subj, expt)
    raw2 = mne.io.read_raw_fif(raw_fname, preload=True)

    raw2.info['dev_head_t'] = raw1.info['dev_head_t'] #overrides kit2fiff head positions, but lets you concatenate raws w/o error
    # also see mne.preprocessing.maxwell_filter for another way to do this

    raw = mne.concatenate_raws([raw1, raw2])
    raw_fname = os.path.join(subj_dir, '%s_%s-raw.fif') % (subj, expt)
    raw.save(raw_fname, overwrite=True)

In [ ]:
### do the following steps by subject
subj = 'subj' 
raw = get_raw_data(subj)
events = mne.find_events(raw, min_duration=0.002)
print('events: ', len(events))
print(raw.info)

In [ ]:
### Filter
print('filtering...')
raw.filter(1,40, method='iir')

In [ ]:
### Plot after filtering (next step: annotate (mark bad time range and channels))
raw.plot()

In [ ]:
### Interpolate bad channels
bad_chs = raw.info['bads']
print('bads: %s'%raw.info['bads'])
print('interpolating bads...')
raw.interpolate_bads()

In [ ]:
### Fitting ica
ica = mne.preprocessing.ICA(n_components=0.95, method='fastica', random_state = 3334) # create ICA object
print('fitting ica...')
ica.fit(raw)

In [ ]:
### Plot ica components
ica.plot_sources(raw) # This plots timecourses of components
ica.plot_components() # This plots topographies of components

In [ ]:
### Confirm which ica components are excluded
ica.exclude

In [ ]:
### Applying ica to raw data
raw = ica.apply(raw, exclude=ica.exclude)
print('saving...')
raw.save('meg/%s/%s_%s-ica-raw.fif' %(subj,subj, sess), overwrite=True)
#pickle.dump(ica.exclude, open('meg/%s/cache/ica_exclude.p'%subj,'wb')) # you can choose to cache excluded components
del raw,ica

In [ ]:
### Load ICA-cleaned data directly
raw_fname = os.path.join(meg_dir, subj, f'{subj}_{sess}-ica-raw.fif')
raw = mne.io.read_raw_fif(raw_fname, preload=True)

In [ ]:
### Define parameters for epoching
# condition name: channel with trigger
event_ids = {
    'event':161, 
}

conditions = event_ids.keys()

# creating the duration of the epochs
epoch_tmin = -0.1           # 100 ms of pre-stimulus blank screen
epoch_tmax = 0.8            # 300 ms of stimulus onscreen, 500 ms stimulus off screen
epoch_baseline = (-0.1,0)   # where to use mne.apply_baseline()

dt = 0.001
decim = 1                   # factor by which you want to downsample your data
onset_delay = 60            # consensus on KIT-NYU MEG photodiode onset delay from trigger (ms)
reject = {'mag': 3e-12}     # reject sensor activity above threshold (radio bursts, not neural data)

In [ ]:
### Creating and rejecting epochs (function)
def make_epochs(subj):
    print(subj)

    cache_dir = join(meg_dir,subj,'cache')
    raw = get_ica_data(subj)
    log = get_log_file(subj)

    picks_meg = mne.pick_types(raw.info, meg=True, eeg=False, eog=False, stim=False)
    pickle_fn = os.path.join(cache_dir,'%s_info.pickled' %subj)
    pickle.dump(raw.info, open(pickle_fn, 'wb'))
    
    events = mne.find_events(raw, min_duration=0.002)
    # print(events)
    events[:,0] += onset_delay
    # print(events)
        
    epochs = mne.Epochs(raw, events, event_id=event_ids,
                    tmin=epoch_tmin, tmax=epoch_tmax,
                    baseline=epoch_baseline, picks=picks_meg,
                    metadata=log, decim=decim, preload=True)

    epochs.apply_baseline(epoch_baseline)
    
    #epochs = #epochs[epochs.metadata["condition"].isin(conditions)] # subset by condition
    #print(epochs)

    print('noise threshold rejection...')
    _len_epochs = len(epochs) # temp variable to calculate rejection count
    epochs.drop_bad(reject)
    n_threshold_rejection = _len_epochs - len(epochs)
    print(n_threshold_rejection, 'dropped')

    return epochs

In [ ]:
### Creating and rejecting epochs
# cache_dir = join(meg_dir,subj,'cache')
raw = get_ica_data(subj)
log = get_log_file(subj)

picks_meg = mne.pick_types(raw.info, meg=True, eeg=False, eog=False, stim=False)
# pickle_fn = os.path.join(cache_dir,'%s_info.pickled' %subj)
# pickle.dump(raw.info, open(pickle_fn, 'wb'))

epochs = make_epochs(subj)
epoch_file_str = '%s_%s-ica-epo.fif' % (subj, expt)
epoch_filename = os.path.join(meg_dir, subj, epoch_file_str)
epochs.save(epoch_filename, overwrite=True) 

In [ ]:
print(np.unique(events[:,2]))  # print all unique event codes in the data
print(raw.annotations) 
events = mne.find_events(raw, min_duration=0.002)
print(events) 

In [ ]:
### Load the saved epoch files
for subj in subjects:
    epoch_file = os.path.join(meg_dir, subj, f'{subj}_{expt}-ica-epo.fif')

    ### Baseline correction
    new_epoch_file = os.path.join(meg_dir, subj, f'{subj}_{expt}-baselinecorr-ica-epo.fif')
    epochs = mne.read_epochs(epoch_file)
    epochs.apply_baseline(epoch_baseline)
    epochs.save(new_epoch_file, overwrite = True) # overwrite old file

In [ ]:
### Plotting epochs according to conditions
cond = 'CONDITION'
evk = epochs[epochs.metadata['condition']==cond].average()
evk.plot_joint(title=cond)